In [ ]:
# Exploring Customer Orders with SQL Aggregation and Joins

  Course: Data Analytics / SQL
  Assignment: Intermediate SQL Querying
  Student Name: RAGUL E
  Date: 13 March 2026

In [1]:
import pandas as pd
import sqlite3

In [2]:
customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

customers_df.head()

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,NaN,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,NaN,05021,Mexico,(5) 555-4729,(5) 555-3745
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,NaN,05023,Mexico,(5) 555-3932,NaN
3,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,NaN,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
4,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,NaN,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


In [3]:
conn = sqlite3.connect(":memory:")

customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")

orders_df.head()

,orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,10248,VINET,5,1996-07-04 00:00:00.000,1996-08-01 00:00:00.000,1996-07-16 00:00:00.000,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NaN,51100,France
1,10249,TOMSP,6,1996-07-05 00:00:00.000,1996-08-16 00:00:00.000,1996-07-10 00:00:00.000,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,NaN,44087,Germany
2,10250,HANAR,4,1996-07-08 00:00:00.000,1996-08-05 00:00:00.000,1996-07-12 00:00:00.000,2,65.83,Hanari Carnes,Rua do Paço 67,Rio de Janeiro,RJ,05454-876,Brazil
3,10251,VICTE,3,1996-07-08 00:00:00.000,1996-08-05 00:00:00.000,1996-07-15 00:00:00.000,1,41.34,Victuailles en stock,2 rue du Commerce,Lyon,NaN,69004,France
4,10252,SUPRD,4,1996-07-09 00:00:00.000,1996-08-06 00:00:00.000,1996-07-11 00:00:00.000,2,51.30,Suprêmes délices,Boulevard Tirou 255,Charleroi,NaN,B-6000,Belgium


In [ ]:
Task 1 — Aggregation and Grouping

In [ ]:
## Task 1 — Aggregation and Grouping

Using the **orders** table, calculate:

- Total number of orders per customer
- Total freight amount
- Average freight amount

Sort the results by **total_freight (descending)** and display the **top 10 rows**.

In [4]:
query1 = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC
"""

result1 = pd.read_sql_query(query1, conn)
result1.head(10)

,customerID,order_count,total_freight,avg_freight
0,SAVEA,31,6683.70,215.603226
1,ERNSH,30,6205.39,206.846333
2,QUICK,28,5605.63,200.201071
3,HUNGO,19,2755.24,145.012632
4,RATTC,18,2134.21,118.567222
5,QUEEN,13,1982.70,152.515385
6,FOLKO,19,1678.08,88.320000
7,BERGS,18,1559.52,86.640000
8,FRANK,15,1403.44,93.562667
9,MEREP,13,1394.22,107.247692


In [ ]:
Task 2 — WHERE vs HAVING

In [ ]:
## Task 2 — WHERE vs HAVING

This task demonstrates the difference between filtering rows **before aggregation (WHERE)** and **after aggregation (HAVING)**.

In [5]:
query2A = """
SELECT
    CustomerID,
    COUNT(OrderID) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID
"""

result2A = pd.read_sql_query(query2A, conn)
result2A.head()

,customerID,high_freight_orders
0,ALFKI,2
1,ANTON,2
2,AROUT,2
3,BERGS,11
4,BLAUS,1


In [6]:
query2B = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500
"""

result2B = pd.read_sql_query(query2B, conn)
result2B.head()

,customerID,total_freight
0,BERGS,1559.52
1,BLONP,623.66
2,BONAP,1357.87
3,BOTTM,793.95
4,EASTC,832.34


In [ ]:
### Explanation

The **WHERE clause** filters rows before aggregation takes place. In Query A, only orders with freight greater than 50 are included before grouping, so the count reflects only high-freight orders.

The **HAVING clause** filters results after aggregation. In Query B, all orders are first grouped by customer and their total freight is calculated. Only customers whose total freight exceeds 500 are then included in the result.

In [ ]:
Task 3 — JOIN and Aggregation

In [ ]:
## Task 3 — JOIN and Aggregation

We will join the **customers** and **orders** tables to analyze customer order activity.

In [7]:
query3_inner = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID
"""

result3_inner = pd.read_sql_query(query3_inner, conn)
result3_inner.head()

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,Berglunds snabbköp,Sweden,18,1559.52


In [8]:
query3_left = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
LEFT JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID
"""

result3_left = pd.read_sql_query(query3_left, conn)
result3_left.head()

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,Berglunds snabbköp,Sweden,18,1559.52


In [ ]:
### Explanation

The INNER JOIN returns only customers who have matching records in the orders table, meaning customers who have placed at least one order.

The LEFT JOIN includes all customers from the customers table, even if they have no orders. When a customer has no orders, the aggregated columns from the orders table appear as NULL.

In [ ]:
## Conclusion

This analysis demonstrates how SQL aggregation, filtering, and joins can be used to analyze customer purchasing behavior. By using GROUP BY, WHERE, HAVING, and JOIN operations, we can derive meaningful insights about order frequency and freight costs across customers.